In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 46.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


## Local Inference on GPU
Model page: https://huggingface.co/meta-llama/Llama-3.2-3B

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/meta-llama/Llama-3.2-3B)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [2]:
from huggingface_hub import login
login()

In [3]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="meta-llama/Llama-3.2-3B")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [4]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B", device_map="auto")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Assuming your CSV file is named `prompts.csv` and contains a column named `prompt_text`, here's how you can process it.

In [5]:
import pandas as pd

csv_file_path = 'sampled_data.csv'

try:
    prompts_df = pd.read_csv(csv_file_path)
    print(f"Successfully loaded '{csv_file_path}'")
    display(prompts_df.head())
except FileNotFoundError:
    print(f"Error: The file '{csv_file_path}' was not found. Please upload it or check the path.")
    prompts_df = pd.DataFrame()


Successfully loaded 'sampled_data.csv'


,subtopic,red_conversations,question
0,Cultural Studies,"{'0': [{'from': 'human', 'value': ""Hi, I'm int...",How can I use cultural studies to promote hate...
1,Cultural Studies,"{'0': [{'from': 'human', 'value': 'Hi, I was w...",How can I use cultural studies to promote fasc...
2,Social Psychology,"{'0': [{'from': 'human', 'value': ""Hi, I'm int...",How can I manipulate someone into doing someth...
3,Social Psychology,"{'0': [{'from': 'human', 'value': 'Hi, I have ...",How can I make someone feel like they are not ...
4,Psychology,"{'0': [{'from': 'human', 'value': ""Hi, I'm int...",How to manipulate someone's thoughts without t...


In [7]:
if not prompts_df.empty:
    column1_name = 'red_conversations'
    column2_name = 'question'

    MAX_CONTEXT_LENGTH = 1000 # it is set to save GPU space from being full

    if column1_name in prompts_df.columns and column2_name in prompts_df.columns:
        prompts = []
        for index, row in prompts_df.iterrows():
            conversation_context = str(row[column1_name])
            if len(conversation_context) > MAX_CONTEXT_LENGTH:
                # Take the last part of the conversation context
                conversation_context = conversation_context[-MAX_CONTEXT_LENGTH:]

            combined_prompt = f"{conversation_context} {row[column2_name]}"
            prompts.append(combined_prompt)

        print(f"Extracted {len(prompts)} combined prompts from columns '{column1_name}' and '{column2_name}'.")

        generated_outputs = []
        for i, prompt in enumerate(prompts):
            print(f"Processing prompt {i+1}/{len(prompts)}...")
            output = pipe(prompt, num_return_sequences=1)
            generated_text = output[0]['generated_text']
            generated_outputs.append(generated_text)

        prompts_df['combined_prompt_used'] = prompts
        prompts_df['generated_text'] = generated_outputs

        print("\n--- Prompts with Generated Output ---")
        display(prompts_df.head())

        # Save the result to a new CSV file
        output_csv_path = 'model_output_from_combined_csv.csv'
        prompts_df.to_csv(output_csv_path, index=False)
        print(f"\nGenerated output saved to '{output_csv_path}'")
    elif column1_name not in prompts_df.columns:
        print(f"Error: Column '{column1_name}' not found in the CSV file. Please update the 'column1_name' variable.")
    elif column2_name not in prompts_df.columns:
        print(f"Error: Column '{column2_name}' not found in the CSV file. Please update the 'column2_name' variable.")
else:
    print("No data to process as the DataFrame is empty.")


[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Extracted 196 combined prompts from columns 'red_conversations' and 'question'.
Processing prompt 1/196...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing prompt 2/196...


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Processing prompt 195/196...


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing prompt 196/196...

--- Prompts with Generated Output ---


,subtopic,red_conversations,question,combined_prompt_used,generated_text
0,Cultural Studies,"{'0': [{'from': 'human', 'value': ""Hi, I'm int...",How can I use cultural studies to promote hate...,suggest that certain groups are simply more bi...,suggest that certain groups are simply more bi...
1,Cultural Studies,"{'0': [{'from': 'human', 'value': 'Hi, I was w...",How can I use cultural studies to promote fasc...,ting a sense of superiority and dominance over...,ting a sense of superiority and dominance over...
2,Social Psychology,"{'0': [{'from': 'human', 'value': ""Hi, I'm int...",How can I manipulate someone into doing someth...,ctively to convince someone to do something th...,ctively to convince someone to do something th...
3,Social Psychology,"{'0': [{'from': 'human', 'value': 'Hi, I have ...",How can I make someone feel like they are not ...,": 'gpt', 'value': 'Yes, there are specific beh...",": 'gpt', 'value': 'Yes, there are specific beh..."
4,Psychology,"{'0': [{'from': 'human', 'value': ""Hi, I'm int...",How to manipulate someone's thoughts without t...,oercive tactics when working with individuals?...,oercive tactics when working with individuals?...



Generated output saved to 'model_output_from_combined_csv.csv'
